In [ ]:
import requests
import pandas as pd
import json
import matplotlib.pyplot as plt

## EDA

In [ ]:
# Import des données bruts
df= pd.read_csv("ghana_health_facilities.csv")
print(df.head(10))


In [ ]:
# Aperçu de la dimension et des types
print(f"Dimensions du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes\n")
df.info()

In [ ]:
# Description des données numériques
df.describe()

# Confidence Score c'est le score de confiance au lieu, s'il existe ou pas encore, 
# vu que c'est des données croisées de 3 sources (OSM, Healthsites et HOT)

In [ ]:
# Nombre de valeurs manquantes par colonne
missing_counts = df.isna().sum()

# Pourcentage de valeurs manquantes
missing_percent = df.isna().mean() * 100

print("Valeurs manquantes par colonne :")
print(missing_counts)

print("\nPourcentage de valeurs manquantes :")
print(missing_percent)


In [ ]:
# description des noms des établissements de santé
df["Facility_Name"].value_counts()

In [ ]:
# description des noms pour les établissements de type "Other"
df[df["Facility_Type"] == "Other"]["Facility_Name"].value_counts()


In [ ]:
#: Inspection des catégories uniques
## Afficher toutes les valeurs uniques et leur fréquence
print("Top 15 des types d'établissements avant nettoyage 😊")
print(df['Facility_Type'].value_counts().head(15))

In [ ]:
#Nous allons créer une fonction pour reclassifier tout ça en 5 grandes catégories standardisées.
def normalize_facility(ftype):
    if pd.isna(ftype):
        return 'Unknown'

    ftype = str(ftype).lower().strip()
    
    # Ordre d'importance pour la classification
    if 'hospital' in ftype:
        return 'Hospital'
    elif 'clinic' in ftype:
        return 'Clinic'
    elif 'chps' in ftype: # Community-based Health Planning and Services
        return 'CHPS'
    elif 'health centre' in ftype or 'health post' in ftype or ftype == 'yes':
        # Hypothèse : on assimile les "Yes" aberrants à des petits postes de santé locaux
        return 'Health Post'
    elif 'pharmacy' in ftype or 'chemical' in ftype or 'dispensary' in ftype:
        return 'Pharmacy'
    else:
        return 'Other'
    
    #Application de la transformation
df['Facility_Category'] = df['Facility_Type'].apply(normalize_facility)

# Vérification du résultat
print("Répartition après nettoyage 😊")
print(df['Facility_Category'].value_counts())

In [ ]:
# df final pour l'analyse et la visualisation
df_final=df
df_final.to_csv("df_final.csv", index=False)

In [ ]:
#pip install seaborn matplotlib

In [ ]:
# Boites à moustaches pour justifier le besoin

from geopy.distance import geodesic

def min_distance_to_hub(row):
    # Coordonnées du centre actuel
    point = (row['Latitude'], row['Longitude'])
    # Calcul de la distance vers chaque hub Zipline
    distances = [geodesic(point, hub['loc']).km for hub in ZIPLINE_HUBS]
    return min(distances)

#On applique le calcul (cela peut prendre 1 min pour 9000 lignes)
df['Dist_to_Nearest_Hub'] = df_final.apply(min_distance_to_hub, axis=1)

#On définit les zones blanches (Beyond 80km)
df_final['Service_Status'] = df_final['Dist_to_Nearest_Hub'].apply(
    lambda x: 'Covered' if x <= 80 else 'Isolated'
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_final, x='Facility_Category', y='Dist_to_Nearest_Hub', palette='viridis')
plt.axhline(y=80, color='r', linestyle='--', label='Limite Zipline (80km)')
plt.title(r"Répartition des distances aux Hubs par type d'établissement")
          
plt.ylabel('Distance au Hub le plus proche (km)')
plt.legend()
plt.show()

In [ ]:
### A faire tourner

# Calculer : combien de villages sont à >30km d'une facility aujourd'hui ? 
# Population non couverte ?

villages_far = df[df["Distance_Nearest_Health_Facility_km"] > 30]
nb_villages_far = len(villages_far)
print("Villages à plus de 30 km d'une facility :", nb_villages_far)

population_non_couverte = villages_far["Population"].sum()
print("Population non couverte :", population_non_couverte)


## Cartographie

In [ ]:
#pip install folium

In [ ]:
import pandas as pd
import folium

# ==============================================================================
# 1. CONFIGURATION DES COULEURS ET DES HUBS
# ==============================================================================

def get_facility_color(facility_type):
    """Retourne une couleur selon le type d'établissement pour l'analyse visuelle."""
    ftype = str(facility_type).lower()
    if 'hospital' in ftype: 
        return '#e74c3c'  # Rouge (Urgence haute)
    if 'clinic' in ftype: 
        return '#3498db'  # Bleu
    if 'chsp' in ftype:
        return '#2ecc71'  # Vert
    if "pharmacy" in ftype:
        return '#f39c12'  # Orange
    else:
        return '#95a5a6'      # Gris (Autres)

# Données réelles des Hubs Zipline
ZIPLINE_HUBS = [
    {"name": "Omenako (GH1)", "loc": [6.2027, -0.4462]},
    {"name": "Mpanya (GH2)", "loc": [7.0545, -1.3855]},
    {"name": "Vobsi (GH3)", "loc": [10.3470, -0.8035]},
    {"name": "Sefwi Wiawso (GH4)", "loc": [6.1820, -2.4845]},
    {"name": "Anum (GH5)", "loc": [6.4885, 0.1625]},
    {"name": "Kete Krachi (GH6)", "loc": [7.7981, -0.0125]}
]

# ==============================================================================
# 2. CRÉATION DE LA CARTE
# ==============================================================================

# Centrage sur le Ghana
m = folium.Map(location=[7.9, -1.0], zoom_start=7, tiles='CartoDB positron')

# Création des groupes de couches (Layer Groups)
fg_health = folium.FeatureGroup(name="Santé (Tous les établissements)")
fg_zipline = folium.FeatureGroup(name="Hubs Zipline (Rayon 40km)")

# --- AJOUT DES 8 900 ÉTABLISSEMENTS DE SANTÉ ---
for _, row in df_final.iterrows():
    popup_info = f"""
    <b>Etablissement:</b> {row['Facility_Name']}<br>
    <b>Type:</b> {row['Facility_Type']}<br>
    """
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=2.5,  # Taille optimale pour voir la densité
        color=get_facility_color(row['Facility_Type']),
        fill=True,
        fill_opacity=0.6,
        popup=folium.Popup(popup_info, max_width=250),
        tooltip=row['Facility_Name']
    ).add_to(fg_health)

# --- AJOUT DES HUBS ZIPLINE ET LEURS RAYONS ---
for hub in ZIPLINE_HUBS:
    # Marqueur Hub
    folium.Marker(
        location=hub["loc"],
        icon=folium.Icon(color='green', icon='plane', prefix='fa'),
        tooltip=f"HUB ZIPLINE: {hub['name']}"
    ).add_to(fg_zipline)
    
    # Cercle de rayon 40km
    folium.Circle(
        location=hub["loc"],
        radius=80000,  # 80 000 mètres = 80 km
        color='green',
        weight=2,
        fill=True,
        fill_color='green',
        fill_opacity=0.1,  # Très léger pour voir les points en dessous
        popup=f"Rayon d'action 40km : {hub['name']}"
    ).add_to(fg_zipline)

# ==============================================================================
# 3. ASSEMBLAGE ET EXPORT
# ==============================================================================

fg_health.add_to(m)
fg_zipline.add_to(m)

# Ajouter le sélecteur de couches (en haut à droite)
folium.LayerControl(collapsed=False).add_to(m)

# Sauvegarde du résultat
m.save('ghana_drone_benchmark.html')

print("Analyse terminée. Fichier 'ghana_drone_benchmark.html' généré.")

In [ ]:
# faire une carte du Ghana avec la densité de population
# faire une carte du Ghana avec des zones de climat
# faire une carte du Ghana avec les reliefs

In [ ]:
import folium
from folium.plugins import HeatMap
import pandas as pd

# Charger ton dataset
df = pd.read_csv("ghana_villages_health_dataset.csv")

# Nettoyage
df = df.dropna(subset=["Latitude", "Longitude", "Population"])
df = df[(df["Latitude"] != 0) & (df["Longitude"] != 0)]

# Préparer les données pour la heatmap
heat_data = df[["Latitude", "Longitude", "Population"]].values.tolist()

# Carte centrée sur le Ghana
m = folium.Map(location=[7.9465, -1.0232], zoom_start=7, tiles="CartoDB positron")

# Heatmap pondérée
HeatMap(
    heat_data,
    radius=20,
    blur=15,
    max_zoom=10,
    min_opacity=0.4
).add_to(m)

m
